# Faster R-CNN 2차 · test 제출 CSV 만들기

학습이 끝난 run 하나의 `weights/best.pt`로 test 842장을 예측해 Kaggle 제출 CSV를 만듭니다. 학습 노트북(`02-cj-faster-rcnn-experiments.ipynb`)과 따로 돌리는 노트북이고, Drive의 run 폴더는 읽기만 하며 CSV는 `rw_runs/submissions/`에 씁니다. **Kaggle에 올리는 것은 직접 합니다.** (하루 10회)

**어디서 실행하나**
- **로컬**: Google Drive for desktop으로 Drive를 연결해 두면 run 폴더를 Drive에서 바로 읽습니다. test 이미지는 레포의 `data/raw/test_images`를 씁니다. Apple M3(MPS)에서 추론에 약 13분 걸립니다.
- **학습 중인 Colab 런타임에 새 커널로 붙이기**: 레포, test 이미지, Drive 연결이 이미 있어서 바로 됩니다. GPU를 학습과 나눠 쓰지만 추론은 test 842장 한 번이라 짧습니다. 이 노트북에서 **런타임 연결 해제·삭제를 하지 않습니다.** 같은 런타임의 학습도 함께 끝납니다.
- **새 Colab GPU 런타임**: 2번 셀이 Drive를 연결하고, 3번 셀이 레포 받기와 `test_images.zip` 풀기를 합니다.

**확인하는 것**
- best.pt가 같은 run의 체크포인트인지 (`config.json`의 signature)
- test 842장을 모두 예측했는지
- 학습 노트북이 run을 끝낼 때 같은 best.pt로 저장한 `test_predictions.json`과 예측이 같은지 (모델을 똑같이 만들었는지)

## 1. 바꿀 값

`RUN_NAME`만 바꿔 쓰면 됩니다. `DRIVE_PATH`는 내 드라이브 안의 경로로, 학습 노트북 1번 셀 `DRIVE_ROOT`에서 `MyDrive/` 뒤 부분과 같습니다.

In [ ]:
from pathlib import Path
import sys

PLATFORM = "colab" if "google.colab" in sys.modules or Path("/content").exists() else "local"

RUN_NAME = "v2_frcnn_e20_s0"  # 제출할 run. 학습이 끝나 weights/best.pt가 있는 폴더 이름

SCORE_MIN = 0.01  # 이 점수 미만 예측은 제출하지 않는다. EXPERIMENTS.md 8단계의 conf=0.01과 같다

BRANCH = "exp/rw-faster-rcnn-exp2"  # 새 Colab 런타임에서 레포를 받을 때만 쓴다

DRIVE_PATH = "CodeIT/Colab/project1-pill-object-detection/pill data"  # 내 드라이브 안에서 zip(data/)과 결과(rw_runs/)가 있는 폴더

if PLATFORM == "colab":
    REPO_ROOT = Path("/content/pill-object-detection")
else:
    REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file()), None)
    if REPO_ROOT is None:
        raise FileNotFoundError("레포 최상위(pyproject.toml)를 찾지 못했습니다. 레포 안에서 노트북을 여세요.")
TEST_DIR = REPO_ROOT / "data/raw/test_images"

print("환경:", PLATFORM, "| 레포:", REPO_ROOT)

## 2. Drive 연결

Colab은 `drive.mount`로 연결합니다. 학습 중인 런타임에 붙였으면 이미 연결돼 있어서 건너뜁니다.

로컬은 [Google Drive for desktop](https://www.google.com/drive/download/)이 연결한 폴더(`~/Library/CloudStorage/GoogleDrive-<계정>/`)에서 `DRIVE_PATH`를 찾습니다. 설치하고 `pill data` 바로가기가 있는 계정으로 로그인해 둡니다. 파일은 읽을 때 받아 오므로 best.pt(약 170MB)는 처음 읽을 때 시간이 조금 걸립니다.

In [ ]:
if PLATFORM == "colab":
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive
        drive.mount("/content/drive")
    drives = [Path("/content/drive/MyDrive")]
else:
    # 계정마다 GoogleDrive-<계정> 폴더가 생기고, 그 아래 내 드라이브 폴더 이름은 언어 설정을 따른다("My Drive", "내 드라이브")
    drives = sorted(p for p in Path.home().glob("Library/CloudStorage/GoogleDrive-*/*") if p.is_dir())
    if not drives:
        raise FileNotFoundError("로컬에 연결된 Google Drive가 없습니다.\n[해결] Google Drive for desktop을 설치하고"
                                "(https://www.google.com/drive/download/) 팀 Drive 계정으로 로그인한 뒤 이 셀을 다시 실행하세요.")
DRIVE_ROOT = next((d / DRIVE_PATH for d in drives if (d / DRIVE_PATH / "rw_runs").is_dir()), None)
if DRIVE_ROOT is None:
    raise FileNotFoundError(f"Drive에서 {DRIVE_PATH}/rw_runs를 찾지 못했습니다. 찾아본 곳: {[str(d) for d in drives]}\n"
                            "[해결] 1번 셀의 DRIVE_PATH를 학습 노트북 1번 셀 DRIVE_ROOT의 MyDrive/ 뒤 경로로 고치세요. "
                            "로컬이면 Finder의 Google Drive에서 pill data 바로가기가 보이는지 확인하세요.")
RUN_DIR = DRIVE_ROOT / "rw_runs" / RUN_NAME
OUT_CSV = DRIVE_ROOT / "rw_runs/submissions" / f"exp2_{RUN_NAME}.csv"
print("run:", RUN_DIR)
print("제출 CSV:", OUT_CSV)

## 3. 환경 준비

Colab에서는 없는 것만 준비합니다. 학습 중인 런타임에 붙였으면 레포와 test 이미지가 이미 있어서 건너뜁니다. 로컬은 `uv sync`로 맞춘 환경과 레포의 test 이미지를 그대로 씁니다.

In [ ]:
import subprocess
import zipfile

if PLATFORM == "colab":
    if not REPO_ROOT.is_dir():
        subprocess.run(["git", "clone", "-q", "--branch", BRANCH, "--single-branch",
                        "https://github.com/codeit-ai-15th/pill-object-detection.git", str(REPO_ROOT)], check=True)
    if not TEST_DIR.is_dir():
        with zipfile.ZipFile(DRIVE_ROOT / "data/test_images.zip") as archive:
            members = [m for m in archive.namelist() if not Path(m).name.startswith("._") and "__MACOSX" not in m]
            archive.extractall(REPO_ROOT, members)
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import torch
import torchvision

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| device", DEVICE)
if DEVICE.type == "cuda":
    free, total = torch.cuda.mem_get_info()
    print(f"GPU {torch.cuda.get_device_name(0)} | 남은 메모리 {free / 1024 ** 3:.1f} / {total / 1024 ** 3:.1f} GB")

## 4. best.pt 불러오기

학습 노트북 `build_model(load_coco=False)`와 같은 모델을 만듭니다. 입력 크기, score·NMS 기준, 최대 검출 수는 run의 `config.json` 값을 그대로 씁니다. BatchNorm은 학습 때처럼 eps 0인 `FrozenBatchNorm2d`로 바꿉니다. 모델 label(1부터, 0은 배경)은 `class_map.csv`로 K-코드 category_id로 바꿉니다.

In [ ]:
import json

import pandas as pd
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import FrozenBatchNorm2d

for name in ("config.json", "class_map.csv", "classes.csv", "weights/best.pt"):
    if not (RUN_DIR / name).is_file():
        raise FileNotFoundError(f"run 파일이 없습니다: {RUN_DIR / name}\n[해결] 1번 셀의 RUN_NAME이 학습을 마친 run인지 확인하세요.")

manifest = json.loads((RUN_DIR / "config.json").read_text())
config = manifest["config"]
class_map = pd.read_csv(RUN_DIR / "class_map.csv")
CATEGORY_OF = {int(label): int(category) for label, category in zip(class_map.model_label, class_map.category_id)}


def freeze_batchnorm(module):
    """BatchNorm2d를 eps 0인 FrozenBatchNorm2d로 바꾼다. 값은 뒤에서 best.pt로 채운다.

    Args:
        module (Module): 바꿀 모델. 자식 모듈까지 바꾼다.
    """
    for name, child in list(module.named_children()):
        if isinstance(child, torch.nn.BatchNorm2d):
            setattr(module, name, FrozenBatchNorm2d(child.num_features, eps=0.0))
        elif isinstance(child, FrozenBatchNorm2d):
            child.eps = 0.0
        else:
            freeze_batchnorm(child)


model = fasterrcnn_resnet50_fpn(weights=None, weights_backbone=None,
                                min_size=config["image_size"], max_size=config["image_size"])
freeze_batchnorm(model)
model.roi_heads.box_predictor = FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features, len(class_map) + 1)
model.roi_heads.score_thresh = config["score_threshold"]
model.roi_heads.nms_thresh = config["nms_iou"]
model.roi_heads.detections_per_img = config["max_detections"]

checkpoint = torch.load(RUN_DIR / "weights/best.pt", map_location="cpu", weights_only=False)
if checkpoint["signature"] != manifest["signature"]:
    raise ValueError("best.pt가 config.json과 다른 실험의 체크포인트입니다. RUN_DIR에 같은 run의 파일만 있는지 확인하세요.")
model.load_state_dict(checkpoint["model"])
model.to(DEVICE).eval()
scores = checkpoint["scores"]
print(f"{RUN_NAME}: best {checkpoint['epoch']} epoch | 클래스 {len(class_map)}개 | "
      f"val {scores['val']:.4f}  kaggle {scores['kaggle']:.4f}  test_only {scores['test_only']:.4f}")

## 5. test 예측

test PNG를 파일 번호 순서로 읽어 FP32로 한 번 예측합니다. 학습 노트북처럼 score 0.001 이상 예측을 모두 남기고, 제출할 때 `SCORE_MIN`으로 거릅니다. bbox는 xyxy를 `xyxy_to_xywh`로 바꿉니다.

In [ ]:
import time

import numpy as np
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from pill_detection.utils import xyxy_to_xywh


class TestImages(Dataset):
    """test PNG를 파일 번호 순서로 읽는다. 정답이 없어서 COCO 파일 대신 폴더를 읽는다."""

    def __init__(self, folder):
        paths = [p for p in Path(folder).glob("*.png") if not p.name.startswith("._")]
        if not paths or not all(p.stem.isdigit() for p in paths):
            raise ValueError(f"test 폴더에 숫자.png 파일이 없거나 다른 이름이 섞였습니다: {folder}\n[해결] test_images.zip을 다시 푸세요.")
        self.paths = sorted(paths, key=lambda p: int(p.stem))

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            array = np.array(image.convert("RGB"), dtype=np.uint8)
        return torch.from_numpy(array).permute(2, 0, 1), self.paths[index].name


dataset = TestImages(TEST_DIR)
if len(dataset) != 842:
    raise ValueError(f"test 이미지가 {len(dataset)}장입니다. 842장이어야 합니다.\n[해결] Colab은 {TEST_DIR}를 지우고 3번 셀을 다시 실행하고, 로컬은 test_images.zip을 레포 최상위에 푸세요.")
loader = DataLoader(dataset, batch_size=config["inference_batch_size"], num_workers=2 if PLATFORM == "colab" else 0,
                    collate_fn=lambda batch: tuple(zip(*batch)))

predictions = []
start = time.perf_counter()
with torch.inference_mode():
    for images, names in loader:
        outputs = model([image.to(DEVICE).float().div(255) for image in images])
        for name, output in zip(names, outputs):
            for box, label, score in zip(output["boxes"].tolist(), output["labels"].tolist(), output["scores"].tolist()):
                predictions.append({"file_name": name, "category_id": CATEGORY_OF[label],
                                    "bbox": xyxy_to_xywh(box), "score": score})
kept = [p for p in predictions if p["score"] >= SCORE_MIN]
print(f"test {len(dataset)}장 예측 {time.perf_counter() - start:.0f}초 | 예측 {len(predictions)}개, "
      f"점수 {SCORE_MIN} 이상 {len(kept)}개 (이미지 {len({p['file_name'] for p in kept})}장)")

## 6. 학습 노트북의 test 예측과 비교

학습 노트북은 run을 끝낼 때 같은 best.pt로 test를 예측해 run 폴더에 `test_predictions.json`으로 저장합니다. 점수 `SCORE_MIN` 이상 예측끼리, 같은 이미지·클래스에서 IoU 0.99 이상인 box가 있으면 같은 예측으로 셉니다. GPU 연산 차이로 점수 기준 근처의 몇 개는 달라질 수 있어서 98% 이상이면 같은 모델로 봅니다. 그보다 낮으면 4번 셀의 모델이 학습 때와 다른 것이므로 제출하지 않고 확인합니다.

In [ ]:
from collections import defaultdict

from torchvision.ops import box_convert, box_iou

cache = RUN_DIR / "test_predictions.json"
if not cache.is_file():
    print("학습 노트북의 test 예측이 없어 비교를 건너뜁니다:", cache)
else:
    def group(rows):
        groups = defaultdict(list)
        for p in rows:
            if p["score"] >= SCORE_MIN:
                groups[p["file_name"], p["category_id"]].append(p)
        return groups

    def xyxy(rows):
        return box_convert(torch.tensor([p["bbox"] for p in rows], dtype=torch.float32), "xywh", "xyxy")

    mine, saved = group(predictions), group(json.loads(cache.read_text()))
    matched, score_gap = 0, 0.0
    for key, rows in mine.items():
        if key not in saved:
            continue
        best_iou, index = box_iou(xyxy(rows), xyxy(saved[key])).max(dim=1)
        for row, iou, j in zip(rows, best_iou.tolist(), index.tolist()):
            if iou >= 0.99:
                matched += 1
                score_gap = max(score_gap, abs(row["score"] - saved[key][j]["score"]))
    n_mine, n_saved = sum(map(len, mine.values())), sum(map(len, saved.values()))
    ratio = matched / max(1, n_mine)
    print(f"점수 {SCORE_MIN} 이상 예측: 이 노트북 {n_mine}개, 학습 노트북 {n_saved}개")
    print(f"같은 이미지·클래스에서 IoU 0.99 이상 겹치는 box {matched}개 ({ratio:.1%}), 점수 차 최대 {score_gap:.4f}")
    print("같은 모델입니다." if ratio >= 0.98 else "[확인 필요] 학습 노트북과 예측이 다릅니다. 제출하지 말고 4번 셀의 모델 설정을 확인하세요.")

## 7. 제출 CSV 쓰기

팀 공용 `write_submission`으로 씁니다. run 폴더의 `classes.csv`에서 대회 클래스(74개)를 읽고, 그 밖의 클래스로 낸 예측은 CSV에서 빠집니다. 같은 이름의 CSV가 있으면 덮어씁니다.

In [ ]:
from IPython.display import display
from pill_detection.evaluate import competition_from_classes, write_submission

competition = competition_from_classes(RUN_DIR / "classes.csv")
rows = write_submission(kept, OUT_CSV, competition)
submission = pd.read_csv(OUT_CSV)
print(f"{rows}줄 저장: {OUT_CSV}")
print(f"이미지 {submission.image_id.nunique()}/842장 | 클래스 {submission.category_id.nunique()}/{len(competition)}개 | "
      f"대회 밖 클래스라 빠진 예측 {len(kept) - rows}개")
display(submission.head())